# HW2 — reproducing *Super-Convergence* (Smith & Topin, 2018) on CIFAR-10 / ResNet-56
Runtime → Change runtime type → **T4 GPU** (or better). Full queue ≈ 3 h on a T4; the main comparison alone ≈ 1 h.

In [ ]:
!nvidia-smi
!pip -q install pyarrow tabulate

Get the code: clone the repo (default) or upload `data.py, model.py, schedules.py, train.py, prepare_data.py, plots.py, run_all.sh` to `/content/hw2`.

In [ ]:
import os
REPO = "https://github.com/rezitdinovAR/efficient-nn.git"
if REPO:
    !git clone -q {REPO} /content/repo
    %cd /content/repo/hw2
else:
    os.makedirs('/content/hw2', exist_ok=True)
    %cd /content/hw2
    need = ['data.py','model.py','schedules.py','train.py','prepare_data.py','plots.py','run_all.sh']
    if not all(os.path.exists(f) for f in need):
        from google.colab import files
        files.upload()
!ls

CIFAR-10 (the Hugging Face parquet mirror is much faster than cs.toronto.edu; pass `torchvision` as argument to use the original tarball).

In [ ]:
!python prepare_data.py

Smoke test (~1 min):

In [ ]:
!python train.py --name smoke --policy 1cycle --lr-min 0.1 --lr-max 3 --mom-max 0.95 --mom-min 0.85 --epochs 1 --eval-every 20 && rm -rf results/smoke

Main comparison only (Fig. 1 of the paper): 1cycle 0.1→3 for 50 epochs vs piecewise-constant LR = 0.35 for 50 / 100 epochs.

In [ ]:
!python train.py --name 1cycle_e50 --policy 1cycle --lr-min 0.1 --lr-max 3 --mom-max 0.95 --mom-min 0.85 --epochs 50
!python train.py --name step_e50 --policy step --lr 0.35 --epochs 50
!python train.py --name step_e100 --policy step --lr 0.35 --epochs 100

Full queue (range tests, Hessian-free LR estimate, limited data, weight-decay / max-LR / Adam ablations, 200-epoch baseline). Finished runs are skipped, so the cell can be re-run after a disconnect.

In [ ]:
!bash run_all.sh

In [ ]:
!python plots.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

In [ ]:
!zip -qr results.zip results -x 'results/*/data*'
from google.colab import files
files.download('results.zip')